# Customer Churn Analysis
**Goal:** Understand why customers leave and build a model to predict churn.

**Sections:**
1. Data Loading & Overview
2. Exploratory Visualization
3. Data Preprocessing
4. Model Development
5. Model Evaluation

## 1. Data Loading & Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

In [ ]:
df = pd.read_csv('churn.csv')

# Fix TotalCharges (has spaces instead of NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print("\nMissing values:", df.isnull().sum().sum())
print("\nChurn distribution:")
print(df['Churn'].value_counts())
print(f"Churn rate: {df['Churn'].eq('Yes').mean():.1%}")

## 2. Exploratory Visualization

### 2a. Churn Rate Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['Churn'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#66c2a5', '#fc8d62'], edgecolor='white')
axes[0].set_title('Churn Count', fontsize=14)
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#66c2a5', '#fc8d62'], startangle=90)
axes[1].set_title('Churn Proportion', fontsize=14)

plt.suptitle('Customer Churn Overview', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2b. Numerical Features vs Churn

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, num_cols):
    for label, color in zip(['No', 'Yes'], ['#66c2a5', '#fc8d62']):
        data = df[df['Churn'] == label][col]
        ax.hist(data, bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
    ax.set_title(col, fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend(title='Churn')

plt.suptitle('Numerical Features by Churn Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 2c. Key Categorical Features vs Churn

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    # Churn rate per category
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean()).sort_values(ascending=False)
    bars = ax.bar(churn_rate.index, churn_rate.values * 100,
                  color=sns.color_palette('Set2', len(churn_rate)), edgecolor='white')
    ax.set_title(f'Churn Rate by {col}', fontsize=13)
    ax.set_ylabel('Churn Rate (%)')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1%}', ha='center', fontsize=10)

plt.suptitle('Churn Rate by Key Categorical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 2d. Tenure vs Monthly Charges (Scatter)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for label, color, marker in zip(['No', 'Yes'], ['#66c2a5', '#fc8d62'], ['o', 'X']):
    subset = df[df['Churn'] == label]
    ax.scatter(subset['tenure'], subset['MonthlyCharges'],
               alpha=0.3, label=f'Churn={label}', color=color, marker=marker, s=15)

ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('Monthly Charges ($)', fontsize=12)
ax.set_title('Tenure vs Monthly Charges by Churn', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 2e. Correlation Heatmap

In [ ]:
# Encode churn and compute correlations with numerical cols
temp = df[num_cols + ['SeniorCitizen']].copy()
temp['Churn'] = (df['Churn'] == 'Yes').astype(int)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(temp.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (Numerical Features)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Drop customer ID (not useful for prediction)
data = df.drop(columns=['customerID'])

# Encode target
data['Churn'] = (data['Churn'] == 'Yes').astype(int)

# Encode all categorical columns with LabelEncoder
le = LabelEncoder()
cat_features = data.select_dtypes(include=['object', 'str']).columns
for col in cat_features:
    data[col] = le.fit_transform(data[col])

print("Features after encoding:")
print(data.dtypes)
data.head()

In [ ]:
X = data.drop(columns=['Churn'])
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"Churn rate (train): {y_train.mean():.1%}")
print(f"Churn rate (test):  {y_test.mean():.1%}")

## 4. Model Development

We train three models and compare them:
- **Logistic Regression** — simple, interpretable baseline
- **Random Forest** — ensemble of decision trees, handles non-linearity
- **Gradient Boosting** — boosted trees, often strong performance

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=3000, solver='saga', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    print(f"✓ {name} trained")

## 5. Model Evaluation

### 5a. Classification Reports

In [ ]:
for name, model in trained.items():
    y_pred = model.predict(X_test)
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    print()

### 5b. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, trained.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=12)

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5c. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#66c2a5', '#fc8d62', '#8da0cb']
for (name, model), color in zip(trained.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 5d. Feature Importance (Best Model)

In [ ]:
# Use Random Forest for feature importance
rf = trained['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))
colors = ['#fc8d62' if v > importances.median() else '#66c2a5' for v in importances.values]
importances.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=14)
ax.set_xlabel('Importance Score')
ax.axvline(importances.median(), color='gray', linestyle='--', alpha=0.7, label='Median')
ax.legend()
plt.tight_layout()
plt.show()

### 5e. Summary Table

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []
for name, model in trained.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1 Score':  f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob),
    })

summary = pd.DataFrame(results).set_index('Model')
summary = summary.round(4)
summary.style.highlight_max(color='lightgreen').format('{:.4f}')

## Key Takeaways

- **Month-to-month contracts** have the highest churn rate — customers on longer contracts are far more loyal.
- **High monthly charges** correlate with churn, especially for Fiber optic users.
- **Short tenure** is a strong churn signal — newer customers are at higher risk.
- **Electronic check** payment method is associated with higher churn.
- **Gradient Boosting** and **Random Forest** outperform Logistic Regression on AUC, making them better choices for production deployment.
- **Top predictive features:** tenure, MonthlyCharges, TotalCharges, Contract, and InternetService.